# RAGAS Test — MERN Stack Question Bank
Yeh notebook RAGBase system ko test karegi MERN PDF ke upar

## Step 1 — Install RAGAS

In [1]:
!pip install ragas datasets -q

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastembed 0.3.3 requires huggingface-hub<1.0,>=0.20, but you have huggingface-hub 1.13.0 which is incompatible.
langchain-experimental 0.0.62 requires langchain-community<0.3.0,>=0.2.6, but you have langchain-community 0.4.1 which is incompatible.
langchain-experimental 0.0.62 requires langchain-core<0.3.0,>=0.2.10, but you have langchain-core 1.3.2 which is incompatible.
langchain-groq 0.1.6 requires langchain-core<0.3,>=0.2.2, but you have langchain-core 1.3.2 which is incompatible.
langchain-qdrant 0.1.1 requires langchain-core<0.3,>=0.1.52, but you have langchain-core 1.3.2 which is incompatible.
streamlit 1.36.0 requires rich<14,>=10.14.0, but you have rich 14.3.4 which is incompatible.
tokenizers 0.19.1 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 1

## Step 2 — Imports

In [2]:
import os
from dotenv import load_dotenv
from pathlib import Path
from datasets import Dataset

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

from langchain_groq import ChatGroq
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from ragbase.ingestor import Ingestor
from ragbase.model import create_llm
from ragbase.retriever import create_retriever
from ragbase.chain import create_chain, ask_question

load_dotenv()
print('Imports done!')

c:\Users\VAISHNAVI\OneDrive\Desktop\rag-visual\ragbase\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\VAISHNAVI\AppData\Local\Temp\ipykernel_23772\3167896363.py:7: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\VAISHNAVI\AppData\Local\Temp\ipykernel_23772\3167896363.py:7: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\VAISHNAVI\AppData\Local\Temp\ipykernel_23772\3167896363.py:7: Deprec

ModuleNotFoundError: No module named 'langchain_core.pydantic_v1'

## Step 3 — PDF Load Karo aur Chain Banao

In [ ]:
# PDF ka path yahan daalo
PDF_PATH = Path("MERNQB.pdf")  # apna PDF yahan rakho

# Ingest karo
print('PDF ingest ho raha hai...')
vector_store = Ingestor().ingest([PDF_PATH])
print('PDF ingest ho gaya!')

# Chain banao
llm = create_llm()
retriever = create_retriever(llm, vector_store=vector_store)
chain = create_chain(llm, retriever)
print('Chain ready!')

## Step 4 — Test Questions (MERN PDF se banaye)

In [ ]:
# Question + Sahi Jawab (ground truth)
test_data = [
    {
        "question": "What is the role of MongoDB in MERN stack?",
        "ground_truth": "MongoDB handles dynamic and unstructured data in the MERN stack."
    },
    {
        "question": "What is the flow of request handling in Express?",
        "ground_truth": "The flow is route → middleware → controller → response."
    },
    {
        "question": "What does the useState hook do in React?",
        "ground_truth": "The useState hook is used to manage component data in React."
    },
    {
        "question": "What is the difference between SQL and MongoDB?",
        "ground_truth": "SQL is a relational database with fixed schema, while MongoDB is a NoSQL database that handles unstructured and dynamic data."
    },
    {
        "question": "How do you integrate frontend with backend in MERN?",
        "ground_truth": "Frontend is integrated with backend using API calls."
    },
    {
        "question": "What is the purpose of middleware in Express?",
        "ground_truth": "Middleware is used in backend development for tasks like logging incoming requests and handling errors."
    },
    {
        "question": "What MongoDB operations can be performed in a Student Management System?",
        "ground_truth": "Insert student records, display all records, update marks of a student, and delete a specific student record."
    },
    {
        "question": "What is the role of CI/CD in deployment?",
        "ground_truth": "CI/CD automates the process of testing and deploying applications in production environments."
    },
]

print(f'Total test questions: {len(test_data)}')

## Step 5 — RAG se Answers Lo

In [ ]:
import asyncio

async def get_answer_and_contexts(question):
    answer = ""
    contexts = []
    async for event in ask_question(chain, question, session_id="ragas-test"):
        if type(event) is str:
            answer += event
        if type(event) is list:
            contexts.extend([doc.page_content for doc in event])
    return answer, contexts

# Saare questions ke answers lo
questions = []
answers = []
contexts = []
ground_truths = []

for item in test_data:
    print(f'Pooch raha hoon: {item["question"][:50]}...')
    answer, context = asyncio.run(get_answer_and_contexts(item["question"]))
    
    questions.append(item["question"])
    answers.append(answer)
    contexts.append(context if context else ["No context found"])
    ground_truths.append(item["ground_truth"])
    print(f'Jawab mila: {answer[:80]}...')
    print('---')

print('\nSaare answers mil gaye!')

## Step 6 — RAGAS Dataset Banao

In [ ]:
ragas_dataset = Dataset.from_dict({
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths,
})

print('Dataset ready!')
print(ragas_dataset)

## Step 7 — RAGAS Evaluate Karo

In [ ]:
# RAGAS ke liye LLM aur Embeddings wrap karo
ragas_llm = LangchainLLMWrapper(ChatGroq(
    temperature=0,
    model_name="llama-3.3-70b-versatile",
    max_tokens=8000
))

ragas_embeddings = LangchainEmbeddingsWrapper(
    FastEmbedEmbeddings(model_name="BAAI/bge-base-en-v1.5")
)

# Evaluate
print('RAGAS evaluation shuru ho rahi hai...')
results = evaluate(
    dataset=ragas_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

print('\nEvaluation complete!')
print(results)

## Step 8 — Results Clearly Dekho

In [ ]:
import pandas as pd

df = results.to_pandas()

print('='*60)
print('RAGAS SCORES — MERN QB RAG TEST')
print('='*60)
print(f'Faithfulness    : {df["faithfulness"].mean():.2f}  (AI ne sirf PDF se jawab diya?)')
print(f'Answer Relevancy: {df["answer_relevancy"].mean():.2f}  (Jawab question se related hai?)')
print(f'Context Precision: {df["context_precision"].mean():.2f} (Sahi chunks mile?)')
print(f'Context Recall  : {df["context_recall"].mean():.2f}  (Koi chunk miss toh nahi?)')
print('='*60)
print('\n0 = Bahut bura | 1 = Perfect')

# Per question results
print('\nPer Question Results:')
pd.set_option('display.max_colwidth', 50)
df[['question', 'faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']]